# 8. Chi-Square Test — Comparing Categorical Distributions

**Building a Heart Disease Risk-Screening System — Notebook 8 of 12, Stage 4: Validating Claims Before Trusting Them**

Notebook 6 tested a proportion; Notebook 7 tested a continuous mean. This notebook
completes the hypothesis-testing toolkit with the third question shape: **is chest
pain type (`cp`) associated with heart disease at all** — a relationship between
two *categorical* variables, where neither a proportion test nor a t-test applies.

## The topic

The chi-square test of independence compares observed counts in a **contingency
table** (a cross-tab of two categorical variables) against the counts you'd expect
if the two variables were unrelated. A large gap between observed and expected is
evidence against independence.

## Why it matters for this system

Several of this registry's most clinically meaningful inputs — chest pain type,
thal, slope — are categorical, not continuous. Notebook 7's toolkit can't touch
them directly. Without this test, the system would either drop these categorical
inputs by default or include them on faith; this notebook is what lets them earn
their place with evidence.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)

## The toolkit

| Tool | Use |
|---|---|
| **Chi-square test of independence** | The default — needs reasonably large expected counts per cell |
| **Fisher's exact test** | Fallback for small tables (2x2) with low expected counts |
| **Cramér's V** | Effect size — how *strong* the association is, not just whether it's significant |
| **Standardized residuals** | Which specific cells drive the overall association |

## How to choose

Start with the chi-square test, but always check the **expected-count rule**
(every cell ≥5) before trusting the result — the test relies on a large-sample
approximation that degrades below that threshold. If it's violated on a 2x2 table,
switch to Fisher's exact test, which computes the exact probability without that
approximation. Whatever test you use, follow it with Cramér's V (statistical
significance and practical strength are different questions, same lesson as Cohen's
h and d in Notebooks 6-7) and residual analysis if you need to know *which*
category is driving the result, not just that one exists.

## Applied to the registry

### Build the contingency table

In [ ]:
contingency = pd.crosstab(df["cp"], df["target"])
contingency.columns = ["no disease", "disease"]
print(contingency)
print("\nDisease rate by chest pain type:")
print((contingency["disease"] / contingency.sum(axis=1)).round(3))

### The chi-square statistic: observed vs. expected

$E$ is what you'd expect in each cell if `cp` and `target` were independent —
computed from the row/column totals alone.

In [ ]:
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"chi-square statistic: {chi2:.2f}")
print(f"degrees of freedom: {dof}")
print(f"p-value: {p_value:.2e}")
print("\nExpected counts under independence:")
print(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns).round(1))

### Expected-count rule: is the test trustworthy here?

In [ ]:
min_expected = expected.min()
print(f"Smallest expected count: {min_expected:.1f}")
print(f"Rule of thumb (>=5) satisfied: {min_expected >= 5}")

### A 2x2 case: `exang` (exercise-induced angina) vs. disease

A binary categorical input is a natural 2x2 table — a good place to compare
chi-square against Fisher's exact test directly.

In [ ]:
exang_table = pd.crosstab(df["exang"], df["target"])
exang_table.index = ["no exercise angina", "exercise angina"]
exang_table.columns = ["no disease", "disease"]
print(exang_table)

chi2_e, p_chi2_e, _, expected_e = stats.chi2_contingency(exang_table)
odds_ratio, p_fisher = stats.fisher_exact(exang_table)

print(f"\nSmallest expected count: {expected_e.min():.1f}")
print(f"Chi-square test: p={p_chi2_e:.2e}")
print(f"Fisher's exact test: odds ratio={odds_ratio:.2f}, p={p_fisher:.2e}")
print("Close agreement here, since expected counts are comfortably above the rule-of-thumb")
print("threshold -- Fisher's exact test matters most exactly when they aren't.")

### Effect size: Cramér's V

In [ ]:
def cramers_v(chi2_stat, table):
    n = table.to_numpy().sum()
    r, k = table.shape
    return np.sqrt((chi2_stat / n) / (min(r - 1, k - 1)))

v_cp = cramers_v(chi2, contingency)
v_exang = cramers_v(chi2_e, exang_table)
print(f"Cramér's V, cp vs target:    {v_cp:.3f}")
print(f"Cramér's V, exang vs target: {v_exang:.3f}")
print("0.1=weak, 0.3=moderate, 0.5+=strong -- both were significant, but this ranks which")
print("categorical input carries the stronger real association.")

### Residual analysis: which specific cells drive the association?

In [ ]:
observed = contingency.to_numpy()
residuals = (observed - expected) / np.sqrt(expected)
residual_df = pd.DataFrame(residuals, index=contingency.index, columns=contingency.columns).round(2)
print("Standardized residuals (cp x target):")
print(residual_df)
print("\n|residual| > 2 flags a cell contributing unusually strongly to the overall association.")

### A categorical screening pass across every candidate categorical input

Before Notebook 9 builds the model, run this test across every categorical
candidate at once — the same filtering instinct as Notebook 5's correlation
ranking, applied to categorical inputs.

In [ ]:
categorical_candidates = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]
results = []
for col in categorical_candidates:
    table = pd.crosstab(df[col], df["target"])
    chi2_c, p_c, _, exp_c = stats.chi2_contingency(table)
    results.append({"input": col, "chi2": round(chi2_c, 1), "p_value": p_c,
                     "cramers_v": round(cramers_v(chi2_c, table), 3),
                     "min_expected_count": round(exp_c.min(), 1)})

results_df = pd.DataFrame(results).sort_values("cramers_v", ascending=False)
print(results_df.to_string(index=False))

## Systems view — what this stage hands to the next one

Stage 4 is complete: Notebooks 6-8 gave the system a validated set of
inputs — continuous and categorical alike — each backed by a p-value, an effect
size, and (where relevant) a check that the test itself was even appropriate to
run. Stage 5 starts now: Notebook 9 assembles these validated inputs into the
system's actual predictive core.

## Try it yourself

1. Check the expected-count rule for every row in Section 8's screening table —
   which candidate inputs (if any) would need Fisher's exact test or a collapsed
   category instead of the standard chi-square test?
2. Compute residuals for the `exang` x `target` table the way Section 7 did for
   `cp` x `target` — which cell contributes most?
3. Using the ranked screening table, decide which 3-4 categorical inputs you'd
   carry into Notebook 9's model, and justify the cut using both p-value and
   Cramér's V, not either alone.